# Demo 9 — Gradient Estimation Attacks
## ZOO (Finite Differences) & Square Attack

**Threat Model:** Score-based black-box — attacker sees **confidence scores**,
no gradients, no architecture knowledge

| Method | Strategy |
|--------|----------|
| **ZOO** | Estimate gradient via finite differences: query `f(x+h*e_i)` and `f(x-h*e_i)` |
| **Square Attack** | Random search in L-inf ball — no gradient estimation needed |

**Why this matters:** Only a small number of API queries is required —
no mathematical knowledge of the model needed.

In [ ]:
# Install
# !pip install torch torchvision matplotlib numpy

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
import torchvision, torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
torch.manual_seed(42); np.random.seed(42)
DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
transform = transforms.Compose([transforms.ToTensor()])
train_set_f = torchvision.datasets.MNIST('./data', train=True, download=True, transform=transform)
test_set_f  = torchvision.datasets.MNIST('./data', train=False, download=True, transform=transform)
train_set = torch.utils.data.Subset(train_set_f, range(1000))
test_set  = torch.utils.data.Subset(test_set_f, range(200))
test_loader = torch.utils.data.DataLoader(test_set, batch_size=256, shuffle=False)

## Step 1 — Train the Victim Model

In [ ]:
class VictimCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Flatten(), nn.Linear(64*7*7,128), nn.ReLU(), nn.Linear(128,10)
        )
    def forward(self, x): return self.net(x)

victim      = VictimCNN().to(DEVICE)
train_loader = torch.utils.data.DataLoader(train_set, batch_size=128, shuffle=True)
opt = optim.Adam(victim.parameters(), lr=1e-3); fn = nn.CrossEntropyLoss()
for epoch in range(3):
    victim.train()
    for X, y in train_loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        opt.zero_grad(); fn(victim(X), y).backward(); opt.step()
victim.eval()
correct = sum((victim(X.to(DEVICE)).argmax(1)==y.to(DEVICE)).sum().item()
               for X, y in test_loader)
print(f'Victim accuracy: {correct/len(test_set)*100:.2f}%')

## Step 2 — ZOO: Finite Difference Gradient Estimation

**Formula:**

$\hat{g}_i = \frac{f(x + h \cdot e_i)[y] - f(x - h \cdot e_i)[y]}{2h}$

This requires **2 x n_pixels queries per step**. We use random coordinate
sampling (10% of pixels) to reduce the query budget.

In [ ]:
def zoo_attack(model, X, y, epsilon=0.2, n_steps=30, h=0.001, lr=0.01,
               coord_sample_rate=0.1):
    """ZOO finite-difference attack on a single image."""
    n_pixels   = X.numel()
    n_coords   = max(1, int(n_pixels * coord_sample_rate))
    x_adv      = X.clone().flatten().to(DEVICE)
    y_true     = y.to(DEVICE)
    total_queries = 0
    for step in range(n_steps):
        coords    = torch.randperm(n_pixels)[:n_coords]
        grad_est  = torch.zeros_like(x_adv)
        for c in coords:
            x_plus  = x_adv.clone(); x_plus[c]  += h
            x_minus = x_adv.clone(); x_minus[c] -= h
            with torch.no_grad():
                s_plus  = torch.softmax(model(x_plus.view(1,1,28,28)),  dim=1)[0, y_true]
                s_minus = torch.softmax(model(x_minus.view(1,1,28,28)), dim=1)[0, y_true]
            grad_est[c] = (s_plus - s_minus) / (2 * h)
            total_queries += 2
        x_adv = x_adv - lr * grad_est.sign()
        delta = (x_adv - X.flatten()).clamp(-epsilon, epsilon)
        x_adv = (X.flatten() + delta).clamp(0, 1)
    return x_adv.view(1, 1, 28, 28), total_queries

## Step 3 — Square Attack: Pure Random Search

Square Attack randomly samples **square-shaped perturbations** and keeps only
those that reduce model confidence. No gradient estimation at all.

In [ ]:
def square_attack(model, X, y, epsilon=0.2, n_queries=200):
    """Simplified Square Attack: random square perturbations in L-inf ball."""
    x_adv  = X.clone().to(DEVICE)
    y_true = y.to(DEVICE)
    _, _, H, W = X.shape
    with torch.no_grad():
        best_conf = torch.softmax(model(x_adv), dim=1)[0, y_true].item()
    for q in range(n_queries):
        size = max(1, np.random.randint(2, max(3, int(H * 0.3))))
        row  = np.random.randint(0, H - size + 1)
        col  = np.random.randint(0, W - size + 1)
        sign = np.random.choice([-1, 1])
        x_candidate = x_adv.clone()
        x_candidate[:, :, row:row+size, col:col+size] += sign * epsilon
        delta       = (x_candidate - X.to(DEVICE)).clamp(-epsilon, epsilon)
        x_candidate = (X.to(DEVICE) + delta).clamp(0, 1)
        with torch.no_grad():
            conf = torch.softmax(model(x_candidate), dim=1)[0, y_true].item()
        if conf < best_conf:
            x_adv = x_candidate; best_conf = conf
    return x_adv

## Step 4 — Evaluate Both Attacks

In [ ]:
n_eval  = 25
epsilon = 0.2
X_eval  = torch.stack([test_set[i][0] for i in range(n_eval)])
y_eval  = torch.tensor([test_set[i][1] for i in range(n_eval)])

# ZOO
zoo_correct = 0; total_zoo_queries = 0; zoo_advs = []
for i in range(n_eval):
    xi, yi  = X_eval[i:i+1], y_eval[i:i+1]
    x_adv, q = zoo_attack(victim, xi, yi, epsilon=epsilon, n_steps=30)
    with torch.no_grad():
        pred = victim(x_adv).argmax(1).item()
    if pred == yi.item(): zoo_correct += 1
    total_zoo_queries += q; zoo_advs.append(x_adv.cpu())
zoo_acc = zoo_correct / n_eval
avg_zoo_queries = total_zoo_queries / n_eval

# Square
sq_correct = 0; sq_advs = []
for i in range(n_eval):
    xi, yi  = X_eval[i:i+1], y_eval[i:i+1]
    x_adv   = square_attack(victim, xi, yi, epsilon=epsilon, n_queries=200)
    with torch.no_grad():
        pred = victim(x_adv).argmax(1).item()
    if pred == yi.item(): sq_correct += 1
    sq_advs.append(x_adv.cpu())
sq_acc = sq_correct / n_eval

print(f'{'Attack':>15}  {'Accuracy':>10}  {'Avg Queries':>13}')
print('-' * 45)
print(f'{'ZOO':>15}  {zoo_acc*100:>9.1f}%  {avg_zoo_queries:>13.0f}')
print(f'{'Square Attack':>15}  {sq_acc*100:>9.1f}%  {'200':>13}')

In [ ]:
fig, axes = plt.subplots(5, 8, figsize=(16, 11))
fig.suptitle(f'Gradient Estimation Attacks (epsilon={epsilon})', fontsize=15, fontweight='bold')
n_show = 8
with torch.no_grad():
    clean_preds = victim(X_eval[:n_show].to(DEVICE)).argmax(1).cpu()
    zoo_preds   = victim(torch.cat(zoo_advs[:n_show]).to(DEVICE)).argmax(1).cpu()
    sq_preds    = victim(torch.cat(sq_advs[:n_show]).to(DEVICE)).argmax(1).cpu()
true = y_eval[:n_show]
clean_imgs = X_eval[:n_show].cpu()
zoo_imgs = torch.cat(zoo_advs[:n_show]).cpu()
sq_imgs = torch.cat(sq_advs[:n_show]).cpu()

# Calculate amplified noise
zoo_noise = (zoo_imgs - clean_imgs) * 5 + 0.5  # amplify by 5x, center at 0.5
sq_noise = (sq_imgs - clean_imgs) * 5 + 0.5

rows = [
    (clean_imgs, clean_preds, 'Clean', 'gray'),
    (zoo_imgs, zoo_preds, f'ZOO (~{avg_zoo_queries:.0f} queries)', 'gray'),
    (zoo_noise.clamp(0, 1), None, 'ZOO Noise (5x)', 'bwr'),
    (sq_imgs, sq_preds, 'Square (200 queries)', 'gray'),
    (sq_noise.clamp(0, 1), None, 'Square Noise (5x)', 'bwr'),
]
for row_idx, (imgs, preds, row_label, cmap) in enumerate(rows):
    for col in range(n_show):
        axes[row_idx, col].imshow(imgs[col].squeeze(), cmap=cmap)
        if preds is not None:
            c = 'green' if preds[col]==true[col] else 'red'
            axes[row_idx, col].set_title(f'P:{preds[col].item()}', color=c, fontsize=10, fontweight='bold')
        axes[row_idx, col].axis('off')
    axes[row_idx, 0].set_ylabel(row_label, fontsize=11, rotation=0, labelpad=90, va='center', fontweight='bold')
plt.tight_layout()
plt.savefig('09_after_attack.png', dpi=150)
plt.show()

def save_row(imgs, title, filename):
    fig, axes = plt.subplots(1, 8, figsize=(16, 2))
    for col in range(8):
        axes[col].imshow(imgs[col].squeeze().cpu(), cmap='gray'); axes[col].axis('off')
    fig.suptitle(title, fontweight='bold')
    plt.tight_layout()
    plt.savefig(filename, dpi=150)
    plt.close()

save_row(X_eval[:8], 'Before Attack: Clean', '09_before_attack.png')
zoo_mid = [zoo_attack(victim, X_eval[i:i+1], y_eval[i:i+1], n_steps=10)[0] for i in range(8)]
save_row(torch.cat(zoo_mid), 'In Process: ZOO (10 steps)', '09_in_process_attack.png')


## Summary

| | ZOO | Square Attack |
|---|---|---|
| **Gradient access** | No | No |
| **Score access** | Yes | Yes |
| **Query efficiency** | Moderate | High |
| **Strength** | Strong | Strong |

**Key takeaway:** Even without gradient access, an attacker who can only see
confidence scores can craft effective adversarial examples.
This undermines 'security through obscurity' of black-box deployment.

**Defenses:** Query rate-limiting, output quantisation, adversarial query detectors.